In [ ]:
import subprocess 
import os

import scienceplots 

import matplotlib.pyplot as plt
import numpy as np
from datasets import load_dataset

import torch
import torch.nn.functional as F
import numpy as np
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from scipy.spatial.distance import cdist
from dotenv import load_dotenv
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from hydra.initialize import initialize_config_dir
from hydra.utils import instantiate
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from skimage.measure import moments, moments_central, inertia_tensor_eigvals
from tqdm import tqdm
from flow_matching.solver import ODESolver
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np


from flowers.models.modules import WrappedModel


load_dotenv()

DATA_ROOT = os.getenv("DATA_ROOT")

In [5]:
data_files = {
    "test": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/test/*.parquet",
    "train": f"{DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0/train/*.parquet",
}
w1 = load_dataset(
    "parquet",
    data_files=data_files
)
keys = ["vae", "uncond", "cond"]
embeddings = {k: np.load(f"./tsne_embeddings/tsne_result_{k}.npy") for k in keys}
embeddings_test = {k: np.load(f"./tsne_embeddings/tsne_result_test_{k}.npy") for k in keys}

In [3]:
# Define the relative path to your config directory
config_dir_relative = "../../src/conf"
config_dir_absolute = os.path.join(os.getcwd(), config_dir_relative)
GlobalHydra.instance().clear()

with initialize_config_dir(config_dir=config_dir_absolute, version_base=None):
    cfg = compose(
        config_name="experiment/rgbmnist_Flow_v2/embed",
        overrides=[
            "++seed=42",
            f"+paths.embedding_dir={DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/embeddings/7518770_0",
            f"+lightning_loader.ckpt_path={DATA_ROOT}/rgbmnist/rgbmnist_Flow_cond_prior/ckpts/7518770_0.ckpt",
            f"++loader.shuffle=False",
        ],
    )

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
raw_dataloader = instantiate(cfg.data.loader)
raw_dataloader.setup()
lightning_loader = instantiate(cfg.lightning_loader)
lightning_loader.to(device)

In [ ]:
def compute_rotation_from_latents(z_vae, model, batch_size=128):
    # Ensure sequential alignment with the t-SNE coordinate indices
    if not isinstance(z_vae, torch.Tensor):
        z_vae = torch.tensor(z_vae)
    
    dataset = TensorDataset(z_vae)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    model.eval()
    results = []

    print("Reconstructing images and measuring geometry...")
    with torch.no_grad():
        for batch in tqdm(loader):
            latent_batch = batch[0].to(next(model.parameters()).device)
            
            # Reconstruction
            z_up = model.projection_up(latent_batch)
            reconstructed_imgs = model.decoder(z_up).cpu().numpy()
            
            for i in range(len(reconstructed_imgs)):
                # Take the first channel (robust for grayscale or RGB-MNIST)
                img = reconstructed_imgs[i, 0] if reconstructed_imgs.ndim == 4 else reconstructed_imgs[i]
                
                # Treat pixels as a distribution to find the principal axis
                M = moments(img, order=1)
                mass = M[0, 0]
                
                if mass <= 0:
                    results.append({"Rotation_Deg": 0, "Eccentricity": 0, "Major_L1": 0})
                    continue
                
                # Physics-based central moments
                centroid = (M[1, 0] / mass, M[0, 1] / mass)
                mu = moments_central(img, center=centroid, order=2)
                
                # Eigenvalues of the inertia tensor
                evals = inertia_tensor_eigvals(img, mu=mu)
                l1, l2 = evals[0], evals[1]
                
                # Calculate the orientation angle theta
                # Formula: theta = 0.5 * atan2(2 * mu_11, mu_20 - mu_02)
                angle = 0.5 * np.degrees(np.arctan2(2 * mu[1, 1], mu[2, 0] - mu[0, 2]))
                
                # Geometric Eccentricity (0 = circle, 1 = line)
                ecc = np.sqrt(1 - (l2 / (l1 + 1e-6)))
                
                results.append({
                    "Rotation_Deg": round(angle, 2),
                    "Eccentricity": round(ecc, 4),
                    "Major_L1": round(l1, 2)
                })

    return pd.DataFrame(results)

dfs = {
    "train": None,
    "test": None
}

for split in ["train", "test"]:
    z_vae = w1[split]["orig"]
    # Execute and synchronize with your existing embeddings
    dfs[split] = compute_rotation_from_latents(z_vae, lightning_loader.base_model)
    dfs[split].to_csv(f"{split}_rotation_aligned.csv", index=False)

In [2]:
# Load dfs if already ran
'''
dfs = {
    "train": None,
    "test": None
}
for split in ["train", "test"]:
    dfs[split] = pd.read_csv(f"{split}_rotation_aligned.csv")
'''

## All Digits

In [6]:
rotation = dfs["train"]["Rotation_Deg"]
coords = embeddings["cond"]

In [ ]:

digits_arr = np.array(w1["train"]["digit"])

v_min, v_max = -90, 90

fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

sc = ax.scatter(
    coords[:, 0], coords[:, 1], 
    c=rotation, 
    marker="o", 
    s=10, 
    cmap='coolwarm', 
    vmin=v_min, 
    vmax=v_max, 
    alpha=0.75
)

ax.set_xlabel(r't-SNE X', fontsize=20, labelpad=10)
ax.set_ylabel(r't-SNE Y', fontsize=20, labelpad=10)

cbar = plt.colorbar(sc, ax=ax, pad=0.02, aspect=30)
cbar.set_label(r'$\hat{\theta}$ (Degrees)', fontsize=20, labelpad=15)
cbar.ax.tick_params(labelsize=10)

cbar.set_ticks([-90, -45, 0, 45, 90])

plt.tight_layout()
plt.show()

# Ones and Sevens

In [ ]:
digits_arr = np.array(w1["train"]["digit"])
mask_1 = (digits_arr == 1)
mask_7 = (digits_arr == 7)

v_min, v_max = -90, 90

fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

sc1 = ax.scatter(
    coords[mask_1, 0], coords[mask_1, 1], 
    c=rotation[mask_1], marker="o", 
    s=15, cmap='coolwarm', 
    vmin=v_min, vmax=v_max, alpha=0.9, label='1'
)

sc7 = ax.scatter(
    coords[mask_7, 0], coords[mask_7, 1], 
    c=rotation[mask_7], marker="^",
    s=15, cmap='coolwarm', 
    vmin=v_min, vmax=v_max, alpha=0.9, label='7'
)

ax.set_xlabel(r't-SNE X', fontsize=36, labelpad=10)
ax.set_ylabel(r't-SNE Y', fontsize=36, labelpad=10)


cbar = plt.colorbar(sc7, ax=ax, pad=0.02, aspect=30)
cbar.set_label(r'$\hat{\theta}$ (Degrees)', fontsize=36, labelpad=15)
cbar.ax.tick_params(labelsize=14)

cbar.set_ticks([-90, -45, 0, 45, 90])

legend = ax.legend(
    title="Digit",
    loc="upper right",
    markerscale=2.5,
    fontsize=24,
    title_fontsize=26,
    frameon=False         
)

plt.tight_layout()
plt.show()

In [ ]:

v_min, v_max = -90, 90 
fig = plt.figure(figsize=(22, 14), dpi=300)

gs = gridspec.GridSpec(2, 2, width_ratios=[1.5, 1], wspace=0.25, hspace=0.3)

ax_main = fig.add_subplot(gs[:, 0])
sc1 = ax_main.scatter(coords[mask_1, 0], coords[mask_1, 1], c=rotation[mask_1], 
                      marker="o", s=20, cmap='coolwarm', vmin=v_min, vmax=v_max, alpha=0.7, label='1')
sc7 = ax_main.scatter(coords[mask_7, 0], coords[mask_7, 1], c=rotation[mask_7], 
                      marker="^", s=20, cmap='coolwarm', vmin=v_min, vmax=v_max, alpha=0.7, label='7')

ax_main.set_title("Combined: Ones and Sevens", fontsize=32, pad=15)
ax_main.set_xlabel(r't-SNE X', fontsize=36)
ax_main.set_ylabel(r't-SNE Y', fontsize=36)
ax_main.legend(title="Digit", loc="upper right", markerscale=2.5, fontsize=24, title_fontsize=26, frameon=False)

ax_1 = fig.add_subplot(gs[0, 1])
ax_1.scatter(coords[mask_1, 0], coords[mask_1, 1], c=rotation[mask_1], 
             marker="o", s=15, cmap='coolwarm', vmin=v_min, vmax=v_max, alpha=0.8)
ax_1.set_title("Digit: 1", fontsize=28)
ax_1.set_ylabel(r't-SNE Y', fontsize=24)
ax_1.set_xlim(ax_main.get_xlim())
ax_1.set_ylim(ax_main.get_ylim())

ax_7 = fig.add_subplot(gs[1, 1])
sc_final = ax_7.scatter(coords[mask_7, 0], coords[mask_7, 1], c=rotation[mask_7], 
                        marker="^", s=15, cmap='coolwarm', vmin=v_min, vmax=v_max, alpha=0.8)
ax_7.set_title("Digit: 7", fontsize=28)
ax_7.set_xlabel(r't-SNE X', fontsize=24)
ax_7.set_ylabel(r't-SNE Y', fontsize=24)
ax_7.set_xlim(ax_main.get_xlim())
ax_7.set_ylim(ax_main.get_ylim())

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.91, 0.15, 0.02, 0.7]) 
cbar = fig.colorbar(sc_final, cax=cbar_ax)
cbar.set_label(r'$\hat{\theta}$ (Degrees)', fontsize=36, labelpad=20)
cbar.set_ticks([-90, -45, 0, 45, 90])
cbar.ax.tick_params(labelsize=18)

plt.show()

# Zeros and Nines

In [ ]:
# 1. Prepare Data
digits_arr = np.array(w1["train"]["digit"])

mask_0 = (digits_arr == 0)
mask_9 = (digits_arr == 9)

v_min, v_max = -90, 90

plt.style.use('science') 
fig, ax = plt.subplots(figsize=(12, 10), dpi=300)

sc0 = ax.scatter(
    coords[mask_0, 0], coords[mask_0, 1], 
    c=rotation[mask_0], marker="o", 
    s=15, cmap='coolwarm', 
    vmin=v_min, vmax=v_max, alpha=0.8, label='0'
)

sc9 = ax.scatter(
    coords[mask_9, 0], coords[mask_9, 1], 
    c=rotation[mask_9], marker="^", 
    s=15, cmap='coolwarm', 
    vmin=v_min, vmax=v_max, alpha=0.8, label='9'
)

ax.set_xlabel(r't-SNE X', fontsize=36, labelpad=10)
ax.set_ylabel(r't-SNE Y', fontsize=36, labelpad=10)


cbar = plt.colorbar(sc9, ax=ax, pad=0.02, aspect=30)
cbar.set_label(r'$\hat{\theta}$ (Degrees)', fontsize=36, labelpad=15)
cbar.ax.tick_params(labelsize=14)

cbar.set_ticks([-90, -45, 0, 45, 90])

legend = ax.legend(
    title="Digit",
    loc="upper right",
    markerscale=2.5, 
    fontsize=24,
    title_fontsize=26,
    frameon=False,
)

plt.tight_layout()
plt.show()